# HuggingFace Transformers: API, Models and Fine-Tuning Techniques - Day 5

In [ ]:
import sys
import torch
import transformers
import PIL

print("Python version:", sys.version)
print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("PIL version:", PIL.__version__)

#### Prompt Engineering Large Language Models in Inference modes

In [36]:
#### Create a dataset of instructions for sentiment classification

def create_instruction_dataset(example, instruction_format="alpaca"):
    text, label = example['text'], example['label']
    if instruction_format == "alpaca":
        instruction = {
            "instruction": "Classify the sentiment of the following review as positive or negative.",
            "input": text,
            "output": "Positive" if label == "1" else "Negative"
        }
    elif instruction_format == "shared_gpt":
        instruction = {
            "conversations": [
                {"from": "user", "value": text},
                {"from": "gpt", "value": "Positive" if label == "1" else "Negative"}
            ],
            "instruction": "Classify the sentiment of the following review as positive or negative."}

# ...
    return instruction    

from csv import DictReader

with open("reviews_simple.csv", "r") as f:
    reader = DictReader(f)
    for row in reader:
        print(create_instruction_dataset(row, instruction_format="shared_gpt"))
        #print(row)

{'conversations': [{'from': 'user', 'value': 'This movie was great!'}, {'from': 'gpt', 'value': 'Positive'}], 'instruction': 'Classify the sentiment of the following review as positive or negative.'}
{'conversations': [{'from': 'user', 'value': 'Excellent storyline'}, {'from': 'gpt', 'value': 'Positive'}], 'instruction': 'Classify the sentiment of the following review as positive or negative.'}
{'conversations': [{'from': 'user', 'value': 'Waste of time and money'}, {'from': 'gpt', 'value': 'Negative'}], 'instruction': 'Classify the sentiment of the following review as positive or negative.'}
{'conversations': [{'from': 'user', 'value': 'Value for money'}, {'from': 'gpt', 'value': 'Positive'}], 'instruction': 'Classify the sentiment of the following review as positive or negative.'}


####  Instruct models

Large Language Models (LLMs) are available in plain form ("base models") and the "Instruct" forms. The Instruct models are suitable for usage as chat-bots. They host a chat instruction template that uses jinja2 template format.

Instructions to them must be first encoded into their chat template format and then tokenized.

In [37]:
#### Load the tokenizer of an LLM and print its chat template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
print(tokenizer.chat_template)


{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [38]:
#### Load the tokenizer of an LLM and print its chat template
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
print(tokenizer.chat_template)


None


In [39]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
print(tokenizer.chat_template)


{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- if messages[0]['content'] is string -%}
        {%- set first_user_prefix = messages[0]['content'] + '

' -%}
    {%- else -%}
        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '

' -%}
    {%- endif -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = "" -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception("Conversation roles must alternate user/assistant/user/assistant/...") }}
    {%- endif -%}
    {%- if (message['role'] == 'assistant') -%}
        {%- set role = "model" -%}
    {%- else -%}
        {%- set role = message['role'] -%}
    {%- endif -%}
    {{ '<start_of_turn>' + role + '
' + (first_user_prefix if loop.first else "") }}
    {%- if message['content'] is string -%}
        {{ message['content'] | trim }}


In [41]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
print(tokenizer.chat_template)


{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for message in messages[::-1] %}
    {%- set index = (messages|length - 

In [ ]:
#### Here is an example of a model that does not have a chat template
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
print(tokenizer.chat_template)


A Typical chat template format (SmolLM2-135M-Instruct)

```
{% for message in messages %}
  {% if loop.first and messages[0]['role'] != 'system' %}
     {{ '<|im_start|>system
         You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
        ' 
     }}
  {% endif %}
 {{'<|im_start|>' + message['role'] + '
    ' + message['content'] + '<|im_end|>' + '
  '}}
{% endfor %}
{% if add_generation_prompt %}
  {{ '<|im_start|>assistant
  ' }}
{% endif %}
```

---
#### Instruction-Tuning Techniques for GPT-style models using ```PEFT``` and ```trl.SFTTrainer```

For supervised fine-tuning, install the package `trl`

```bash
!pip install trl
```

`trl` provides `SFTTrainer` and `SFTConfig` for training using supervised fine-tuning.


In [ ]:
!pip install trl

In [42]:
import trl
trl.__version__

'1.10.0'

---

#### Evaluate a small GPT-style language model `SMolLM2-135M-Instruct` before fine-tuning

In [46]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator

device = Accelerator().device

#model_name = "HuggingFaceTB/SmolLM2-135M"
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

# Models with -Instruct suffix are instruction-tuned and support chat templates. 
# You can use them with the `apply_chat_template` method of the tokenizer.

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
).to(device)

print("Model and Tokenizer loaded successfully.")

# prepare the model input
prompt = "Can you tell me more about Jagadish Chandra Bose?"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print("Prepared input text:", text)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model and Tokenizer loaded successfully.
Prepared input text: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Can you tell me more about Jagadish Chandra Bose?<|im_end|>
<|im_start|>assistant



In [47]:

model_inputs = tokenizer([text], return_tensors="pt").to(device)
print("Model input:", model_inputs)
print("Model input prepared successfully.")
print("-" * 50)

# Generate the output
generated_ids = model.generate(**model_inputs, max_new_tokens=128)
print("Output generated successfully.")
print("-" * 50)
print("Generated IDs:", generated_ids)

Model input: {'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  5356, 11173,    30,     2,
           198,     1,  4093,   198,  7306,   346,  2505,   549,   540,   563,
         47029, 44615, 34045,   389,   602,    47,     2,   198,     1,   520,
          9531,   198]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}
Model input prepared successfully.
--------------------------------------------------
Output generated successfully.
--------------------------------------------------
Generated IDs: tensor([[    1,  9690,   198,  2683,   359,   253,  5356, 11173,    30,     2,
           198,     1,  4093,   198,  7306,   346,  2505,   549,   540,   563,
         47029, 44615, 34045,   389,   602,    47,     2,   198,     1,   520,
          9531,   198,    58,   454, 44615, 34045,   389,   602,   436,   253,
         14104,  3530, 21130,   284, 27319, 

In [48]:

# Get and decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
print(tokenizer.decode(output_ids, skip_special_tokens=True))


Jagadish Chandra Bose was a renowned Indian physicist and chemist who made significant contributions to the field of physics. Born in 1857 in Calcutta, India, Bose was a close friend of Lord Williamṃ C. Maxwell, the father of Albert Einstein. Bose's work on electromagnetic theory and his work on the theory of light was groundbreaking and influential.

Bose's work on electromagnetic theory, particularly his work on the theory of electromagnetic waves, laid the foundation for the development of modern electromagnetic theory. He was also a pioneer in the field of quantum mechanics, particularly in the theory of atoms and molecules.



In [49]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator

device = Accelerator().device

model_name = "HuggingFaceTB/SmolLM2-135M"

# 1. Load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
).to(device)

print("Model and Tokenizer loaded successfully.")

# As models without the -Instruct suffix are not instruction-tuned, 
# they do not support chat templates.
# We need to define a chat template and add special tokens to the tokenizer.

# 2. Define the ChatML Jinja2 template string
chat_template = (
    "{% for message in messages %}"
    "<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

# 3. Add special chat boundaries if they aren't in the vocabulary
special_tokens = {"additional_special_tokens": ["<|im_start|>", "<|im_end|>"]}
tokenizer.add_special_tokens(special_tokens)

# Resize model embeddings to match the new token configuration
model.resize_token_embeddings(len(tokenizer))

# 4. Assign the template to the tokenizer
tokenizer.chat_template = chat_template

# 5. Verify the configuration
prompt = "Who is Chandrashekar Babu?"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(device)

print("Model input prepared successfully.")

# Generate the output
generated_ids = model.generate(**model_inputs, max_new_tokens=150)

# Get and decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
print(tokenizer.decode(output_ids, skip_special_tokens=True))


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model and Tokenizer loaded successfully.
Model input prepared successfully.
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking
Who is Chandrashekar Babu?,,,,,,,,,,,,,,,,aking
,,,,,,,,,,,,,,,,aking



#### Now let's fine-tune the models with our custom biography dataset

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir ../../data/logs/

In [ ]:
pwd

In [ ]:
model


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, TaskType
from accelerate import Accelerator
device = Accelerator().device

# 1. Setup Model and Tokenizer
# Using the 135M-Instruct parameter model as a fast, lightweight starter
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load model in bfloat16 for fast training
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.bfloat16, 
).to(device)

# 2. Configure LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 3. Load Your Local Biography Dataset
dataset = load_dataset("json", data_files="./chandra_biography_instruct.jsonl", split="train")

# 4. Define Training Arguments
training_args = SFTConfig(
    output_dir="../../data/smollm2-biography",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,            # Optimized for the 135M model
    num_train_epochs=25,           # High epochs to ensure memorization
    bf16=True,                     # Use bfloat16 for faster training on compatible hardware
    save_strategy="no",            # Disable saving checkpoints to save disk space
    dataset_text_field="text",
    dataloader_pin_memory=False,

    report_to="trackio",                    # Use trackio for logging
    project="smollm2-prompt-tuning",   # Name of the project for logging
    run_name="smollm2-biography-1",

    logging_steps=1,
)

# 5. Initialize Trainer with updated arguments
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,    # Replaces 'tokenizer=' to avoid deprecation warnings
    args=training_args,            # Pass the SFTConfig object here
)
# 6. Run Training and Save
print("Starting training...")
trainer.train()
trainer.model.save_pretrained("../../data/SmolLM2-135M-Instruct-Biography")
print("Training complete! LoRA weights saved.")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Starting training...
* Resumed existing run: smollm2-biography-1


/opt/anaconda3/envs/hf5_env_new/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,3.280772
2,3.269407
3,2.989205
4,2.762267
5,2.512334
6,2.434992
7,2.225568
8,2.018678
9,2.093736
10,1.936362


/opt/anaconda3/envs/hf5_env_new/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/hf5_env_new/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/hf5_env_new/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/hf5_env_new/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/hf5_env_new/lib/pyth

* Run finished. Uploading logs to Trackio (please wait...)
Training complete! LoRA weights saved.


In [51]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
from accelerate import Accelerator
device = Accelerator().device

base_model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id, 
                dtype=torch.bfloat16, 
)

# Merge your biography LoRA weights
model = PeftModel.from_pretrained(base_model, "../../data/SmolLM2-135M-Instruct-Biography")
model = model.to(device)

# prepare the model input
prompt = "Tell me more about Chandrashekar Babu."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(device)

outputs = model.generate(**model_inputs, max_new_tokens=150, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

system
You are a helpful assistant.
user
Tell me more about Chandrashekar Babu.
assistant
Chandrashekar Babu is a FOSS Technologist and trainer based in Chennai, India.


In [53]:
# prepare the model input
prompt = "Where does Chandrashekar Babu work?"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(device)

outputs = model.generate(**model_inputs, max_new_tokens=150, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


system
You are a helpful assistant.
user
Where does Chandrashekar Babu work?
assistant
Chandrashekar Babu works in Chennai, India.


---
#### Fine-tuning the base model

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from accelerate import Accelerator
device = Accelerator().device

model_id = "HuggingFaceTB/SmolLM2-135M" 

# 1. Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.bfloat16, 
).to(device)

# 2. Configure LoRA
peft_config = LoraConfig(
    r=8, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

# 3. Load the structured dataset
dataset = load_dataset("json", data_files="chandra_biography_instruct.jsonl", split="train")

# 4. Use SFTConfig to inject the Chat Template
training_args = SFTConfig(
    output_dir="../../data/SmolLM2-135M-Biography-Chat",  # Directory to save the trained model
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=6e-4,
    num_train_epochs=30,
    bf16=True,
    save_strategy="no",

    report_to="trackio",    # Report to TrackIO for experiment tracking
    logging_steps=1,
    run_name="SmolLM2-Biography-Chat-3", # Name of the run for tracking

    dataset_text_field="text",
    # MAGIC HAPPENS HERE: Copies the official chat format from the Instruct variant
    chat_template_path="HuggingFaceTB/SmolLM2-135M-Instruct" 
)

# 5. Initialize Trainer (It will auto-detect the "messages" column)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Training with Chat Template...")
trainer.train()
trainer.model.save_pretrained("../../data/SmolLM2-135M-Biography-Chat")
tokenizer.save_pretrained("../../data/SmolLM2-135M-Biography-Chat") # Save tokenizer to preserve template
print("Done!")


In [ ]:
pwd

In [ ]:
ls ../../data/unsloth-SmolLM2-135M-Instruct-Biography

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
from accelerate import Accelerator
device = Accelerator().device

base_model_id = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id, 
                dtype=torch.bfloat16, 
)

# 2. Define the ChatML Jinja2 template string
chat_template = (
    "{% for message in messages %}"
    "<|im_start|>{{ message['role'] }}\n{{ message['content'] }}<|im_end|>\n"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

# 3. Add special chat boundaries if they aren't in the vocabulary
special_tokens = {"additional_special_tokens": ["<|im_start|>", "<|im_end|>"]}
tokenizer.add_special_tokens(special_tokens)

# Resize model embeddings to match the new token configuration
base_model.resize_token_embeddings(len(tokenizer))

# 4. Assign the template to the tokenizer
tokenizer.chat_template = chat_template


adapter = "../../data/SmolLM2-135M-Biography-Chat"

# Merge your biography LoRA weights
model = PeftModel.from_pretrained(base_model, adapter)
model = model.to(device)

# prepare the model input
prompt = "Can you tell me more about Chandrashekar Babu?"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(device)

outputs = model.generate(**model_inputs, max_new_tokens=150, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "../../data/unsloth_SmolLM2"

model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to("mps")
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Where does Chandrashekar Babu work?"
messages =[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to("mps")
generated_ids = model.generate(**model_inputs, max_new_tokens=150)

output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(output)

---

### RAG Overview

RAG (Retrieval-Augmented Generation) is an approach that combines pre-trained language models with external knowledge sources to improve the quality and relevance of generated content. 

Instead of relying solely on the information encoded in the model's parameters, RAG systems retrieve relevant documents or data from a knowledge base or database at inference time. 

This retrieved information is then used to inform and enhance the model's responses.

#### How RAG works
- **Retrieval:** When a user asks a question, the RAG system performs a search on an external knowledge base to find information that is relevant to the query. This knowledge base can include anything from the latest internet data to a company's internal documents.

- **Augmentation:** The retrieved information is then added to the original prompt given to the LLM.

- **Generation:** The LLM generates a response that is now grounded in the information it was just given, making the answer more accurate, specific, and relevant.

##### Key benefits
- **More accurate and relevant answers:** RAG provides LLMs with current and specific data, leading to higher-quality outputs.

- **Cost-effective:** It offers a way to update LLM knowledge with new information without the high cost and effort of retraining the entire model. 

- **Context-aware:** RAG allows AI to access domain-specific or proprietary information, enabling it to answer questions about a particular company or industry.

- **Reduces hallucinations:** By providing factual context, RAG helps to ground the LLM's responses and reduce the likelihood of making things up. 

#### Key components of a RAG system
 - **Retrieval Model**: fetches relevant information from an external knowledge source - a database, search engine of other forms of knowledge repository.
 - **Language Model**: generates responses based on the retrieved knowledge.

#### A Typical RAG implementation workflow

Implementing a typical RAG worflow involves the following components:
- **Embedding model:** A pre-trained language model that converts input text into embeddings - vector representations that capture semantic meaning. These vectors will be used to search for relevant information in the dataset.

- **Vector database:** A storage system for knowledge and its corresponding embedding vectors. While there are many vector database technologies like Qdrant, Pinecone, and pgvector, or even simple in-memory database.

- **Chatbot:** A language model that generates responses based on retrieved knowledge. This can be any language model, such as Llama, Gemma, or GPT.

Two vital steps are involved to setup these components:

- **Indexing Phase**: breaking the dataset (or documents) into small chunks and calculating a vector representation for each chunk that can be efficiently searched during generation.
<div style="background-color: #546a51; display: inline-block; padding: 10px;">
  <img src="./rag_indexing.svg" alt="RAG Indexing">
</div>

The size of each chunk can vary depending on the dataset and the application. For example, in a document retrieval system, each chunk can be a paragraph or a sentence. In a dialogue system, each chunk can be a conversation turn.

After the indexing phrase, each chunk with its corresponding embedding vector will be stored in the vector database.

The embedding vectors can be later used to retrieve relevant information based on a given query. Think of it as SQL WHERE clause, but instead of querying by exact text matching, we can now query a set of chunks based on their vector representations.

To compare the similarity between two vectors, we can use cosine similarity, Euclidean distance, or other distance metrics. Here is the formula for cosine similarity between two vectors A and B:

![](./cosine_similarity.webp)

---

- **Retrieval Phase**: calculate the Query Vector to represent the query, and compare it against the vectors in the database to find the most relevant chunks.

The result returned by The Vector Database will contains top N most relevant chunks to the query. These chunks will be used by the Chatbot to generate a response.
<div style="background-color: #546a51; width=150px;">
  <img src="./rag_retrieval_phase.svg" alt="RAG Retrieval Phase">
</div>


### Simple RAG Implementation Demo - 1

In [54]:
from dotenv import load_dotenv
load_dotenv("../../dotenv.sh")
import os
OWM_API_KEY = os.getenv("OWM_API_KEY")

In [55]:
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


In [57]:

def get_weather(city: str) -> dict:
    import requests
    BASE_URL = "http://api.openweathermap.org/data/2.5/weather"
    API_KEY = OWM_API_KEY
    QUERY_STRING = f"?q={city}&units=metric&APPID={API_KEY}"
    URL = BASE_URL + QUERY_STRING

    response = requests.get(URL)
    if response.ok:
        return response.json()
    else:
        return {"error": "Could not fetch weather data."}
    

get_weather("Chennai")

{'coord': {'lon': 80.2785, 'lat': 13.0878},
 'weather': [{'id': 803,
   'main': 'Clouds',
   'description': 'broken clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 36.12,
  'feels_like': 43.12,
  'temp_min': 34.99,
  'temp_max': 36.67,
  'pressure': 1007,
  'humidity': 50,
  'sea_level': 1007,
  'grnd_level': 1006},
 'visibility': 10000,
 'wind': {'speed': 3.6, 'deg': 300},
 'clouds': {'all': 82},
 'dt': 1787294796,
 'sys': {'type': 2,
  'id': 2111253,
  'country': 'IN',
  'sunrise': 1787272037,
  'sunset': 1787317041},
 'timezone': 19800,
 'id': 1264527,
 'name': 'Chennai',
 'cod': 200}

In [60]:


# 1. Define the actual local tool function
def get_weather_old(city: str) -> str:
    # Simulated API response
    if "copenhagen" in city.lower():
        return json.dumps({"temperature": "12°C", "condition": "Rainy"})
    return json.dumps({"temperature": "20°C", "condition": "Sunny"})

# 2. Setup model and tokenizer
checkpoint = "HuggingFaceTB/SmolLM3-3B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint, dtype=torch.bfloat16).to("mps")

# 3. Define the tool schema and starting conversation
tools = [
    {
        "name": "get_weather",
        "description": "Get current weather info for a city",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"]
        }
    }
]

messages = [{"role": "user", "content": "What is the current weather in Pune?"}]

# ==========================================
# STEP 1: Generate the Tool Call
# ==========================================
inputs = tokenizer.apply_chat_template(
    messages, 
    xml_tools=tools, 
    add_generation_prompt=True,
    #tokenize=False, 
    return_tensors="pt"
).to("mps")
inputs

r = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
print(r)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 21 August 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous steps. In 

In [ ]:
print(inputs)

In [61]:
import re
import json
outputs = model.generate(**inputs, max_new_tokens=128)
assistant_text = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True).strip()

print(f"--- Model Output ---\n{assistant_text}\n")


--- Model Output ---
<think>I need to retrieve the current weather information for Pune from the available tools. The `get_weather` function is the appropriate tool for this task. I will call this function with the city parameter set to "Pune".</think>
<tool_call>
{"name": "get_weather", "arguments": {"city": "Pune"}}
</tool_call>



In [62]:

# Append model's response to conversation history
messages.append({"role": "assistant", "content": assistant_text})

# ==========================================
# STEP 2: Parse and Execute the Tool
# ==========================================
# Extract contents inside <tool_call>...</tool_call>
match = re.search(r"<tool_call>(.*?)</tool_call>", assistant_text, re.DOTALL)

if match:
    tool_call_data = json.loads(match.group(1).strip())
    tool_name = tool_call_data.get("name")
    tool_args = tool_call_data.get("arguments", {})

    print(f"--- Executing Tool: {tool_name} with args {tool_args} ---")
    
    # Execute the mapped function
    if tool_name == "get_weather":
        tool_result = get_weather(**tool_args)
    else:
        tool_result = "Error: Tool not found."

    print(f"Tool Result: {tool_result}\n")

    # Append the result back using the 'tool' role
    messages.append({
        "role": "tool",
        "name": tool_name,
        "content": tool_result
    })

    # ==========================================
    # STEP 3: Generate Final Answer
    # ==========================================
    final_inputs = tokenizer.apply_chat_template(
        messages, 
        xml_tools=tools, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to(model.device)

    final_outputs = model.generate(**final_inputs, max_new_tokens=512)
    final_answer = tokenizer.decode(final_outputs[0][len(final_inputs[0]):], skip_special_tokens=True)

    print(f"--- Final Answer ---\n{final_answer.strip()}")
else:
    print("Model did not request a tool call.")


--- Executing Tool: get_weather with args {'city': 'Pune'} ---
Tool Result: {'coord': {'lon': 73.8553, 'lat': 18.5196}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 28.24, 'feels_like': 28.19, 'temp_min': 27.45, 'temp_max': 29.21, 'pressure': 1010, 'humidity': 44, 'sea_level': 1010, 'grnd_level': 938}, 'visibility': 10000, 'wind': {'speed': 7.11, 'deg': 267, 'gust': 8.81}, 'clouds': {'all': 51}, 'dt': 1787295646, 'sys': {'type': 2, 'id': 2107474, 'country': 'IN', 'sunrise': 1787273267, 'sunset': 1787318894}, 'timezone': 19800, 'id': 1259229, 'name': 'Pune', 'cod': 200}

--- Final Answer ---
<think>The response from the `get_weather` function indicates that the current weather in Pune includes a temperature of 31.0°C, a relative humidity of 86%, and a wind speed of 10.0 km/h with a wind direction of North. I will provide this information to the user.</think>
<tool_call>
{"name": "get_weather_response", "c

In [63]:
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Define local mock functions
def get_weather(city: str) -> str:
    weather_db = {
        "copenhagen": {"temperature": "12°C", "condition": "Rainy"},
        "tokyo": {"temperature": "22°C", "condition": "Clear"},
    }
    res = weather_db.get(city.lower(), {"temperature": "20°C", "condition": "Sunny"})
    return json.dumps(res)

# 2. Setup model and tokenizer
checkpoint = "HuggingFaceTB/SmolLM3-3B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.bfloat16, device_map="auto")

# 3. Define schema for the tool
tools = [
    {
        "name": "get_weather",
        "description": "Get current weather info for a city",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"]
        }
    }
]

# Ask a query requiring multiple lookups
messages = [{"role": "user", "content": "Compare the weather between Copenhagen and Tokyo right now."}]

# ==========================================
# STEP 1: Generate Thinking + Tool Calls
# ==========================================
# We enable 'enable_thinking=True' so the model plans its tool execution first
inputs = tokenizer.apply_chat_template(
    messages, 
    xml_tools=tools, 
    enable_thinking=True, 
    add_generation_prompt=True, 
    return_tensors="pt"
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)
assistant_text = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()

print(f"--- Raw Model Output ---\n{assistant_text}\n")
messages.append({"role": "assistant", "content": assistant_text})

# ==========================================
# STEP 2: Extract Thinking and Tool Calls
# ==========================================
# Extract thinking block if present
think_match = re.search(r"<think>(.*?)</think>", assistant_text, re.DOTALL)
if think_match:
    print(f"--- Model's Internal Thought Process ---\n{think_match.group(1).strip()}\n")

# Extract ALL tool calls using re.findall
tool_calls = re.findall(r"<tool_call>(.*?)</tool_call>", assistant_text, re.DOTALL)

if tool_calls:
    print(f"--- Found {len(tool_calls)} Parallel Tool Calls ---")
    
    # Process each tool call sequentially or concurrently
    for call_str in tool_calls:
        tool_call_data = json.loads(call_str.strip())
        tool_name = tool_call_data.get("name")
        tool_args = tool_call_data.get("arguments", {})

        print(f"Executing: {tool_name}(city='{tool_args.get('city')}')")
        
        if tool_name == "get_weather":
            tool_result = get_weather(**tool_args)
        else:
            tool_result = "Error: Tool not found."

        # Append EACH individual tool result back to history
        messages.append({
            "role": "tool",
            "name": tool_name,
            "content": tool_result
        })
    print()

    # ==========================================
    # STEP 3: Generate the Final Answer
    # ==========================================
    final_inputs = tokenizer.apply_chat_template(
        messages, 
        xml_tools=tools, 
        enable_thinking=False, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to(model.device)

    final_outputs = model.generate(**final_inputs, max_new_tokens=256)
    final_answer = tokenizer.decode(final_outputs[0][len(final_inputs.input_ids[0]):], skip_special_tokens=True)

    print(f"--- Final Answer ---\n{final_answer.strip()}")
else:
    print("Model answered directly without tools.")


Loading weights:   0%|          | 0/326 [00:03<?, ?it/s]

--- Raw Model Output ---
<think>I need to retrieve the current weather data for both Copenhagen and Tokyo. I'll use the get_weather function for each city. The arguments for each function will include the city name as 'Copenhagen' and 'Tokyo' respectively. Once I have the data, I can compare the temperature, weather conditions, and other relevant details to provide the comparison.</think>
<tool_call>
{"name": "get_weather", "arguments": {"city": "Copenhagen"}}
</tool_call>
<tool_call>
{"name": "get_weather", "arguments": {"city": "Tokyo"}}
</tool_call>

--- Model's Internal Thought Process ---
I need to retrieve the current weather data for both Copenhagen and Tokyo. I'll use the get_weather function for each city. The arguments for each function will include the city name as 'Copenhagen' and 'Tokyo' respectively. Once I have the data, I can compare the temperature, weather conditions, and other relevant details to provide the comparison.

--- Found 2 Parallel Tool Calls ---
Executing:

---

#### Using ```ollama```

We can also use models hosted via `ollama` server locally.

In [ ]:
!ollama pull hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
!ollama pull hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF

In [ ]:
!ollama list

In [ ]:
# Use a sample document for RAG demo
!curl -o cat-facts.txt https://huggingface.co/ngxson/demo_simple_rag_py/raw/main/cat-facts.txt


In [ ]:
!head -n 5 cat-facts.txt

In [ ]:
### Load the dataset
dataset = []
with open("cat-facts.txt", "r") as ins:
    dataset = ins.readlines()
print(f"Loaded {len(dataset)} entries for RAG demo.")

In [ ]:
%pip install ollama

In [ ]:
### Implement the vector database

import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))

for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')


In [ ]:
VECTOR_DB[0]

In [ ]:
### Implement the similarity search
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

In [ ]:
# Retrieve top N relevant chunks for a given query

def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]


In [ ]:
# Generation with retrieved context

#input_query = input('Ask me a question: ')
input_query = "How long do cats sleep every day on average?"
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''


In [ ]:
# # Stream the response from the chatbot
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)


---

### Using FAISS

FAISS (Facebook AI Similarity Search), is an open-source library for performing fast similarity searches and clustering on high-dimensional vectors. Developed by Meta AI, it provides algorithms that can handle datasets of any size, including those that don't fit into RAM, and is used in applications like image/video retrieval, recommendation systems, and natural language processing. 

FAISS is written in C++ but has Python bindings, and it includes both CPU and GPU-accelerated implementations.  

#### Key features of FAISS
 - **Similarity Search:** Efficiently finds the closest matching vectors to a query vector within a large dataset. 
 - **Clustering:** Groups similar vectors together. 
 - **Handles large datasets:** Can search and cluster billions of vectors, even those exceeding available RAM. 
 - **Vector Compression:** Includes various methods to compress vectors, which reduces memory usage. 
 - **Both CPU and GPU support:** Provides high-performance implementations that can leverage the power of GPUs, particularly NVIDIA GPUs from 2012 onwards. 
 - **Indexing methods:** Offers different types of indexes to optimize search performance, such as Flat indexes for brute-force search and Inverted File indexes for faster, approximate searches by clustering vectors first. 

#### Common uses
 - **Recommendation systems:** Finding similar items or users based on their embeddings. 
 - **Image and video retrieval:** Searching for visually similar images or video clips. 
 - **Natural Language Processing (NLP):** Finding semantically similar text documents or sentences. 


---

### RAG demo using FAISS

In [ ]:
%pip install sentence-transformers sentencepiece faiss-cpu

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# --- PART 1: THE RETRIEVER (Knowledge Base Setup) ---

# 1. Our "Private" Knowledge Base (Facts the model might not know or could hallucinate)
documents = [
    "The secret code to unlock the server is 99-ALPHA-BETA.",
    "Project Apollo was launched in 2024 to simulate Mars habitats in Arizona.",
    "The CEO of TechCorp is currently Sarah Connor, appointed in late 2023.",
    "Python 4.0 is rumored to be released in 2030 with a new JIT compiler."
]

print("Indexing documents...")

# 2. Load an Embedding Model (Small, fast, standard for RAG)
# This model converts text into a vector of 384 numbers.
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Create Embeddings for our documents
doc_embeddings = embedder.encode(documents)

# 4. Build the Vector Index using FAISS
# We use a simple L2 (Euclidean) distance index.
# dimension = 384 (output size of MiniLM)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print(f"Indexed {index.ntotal} documents.\n")



Google **Flan-T5-base** is a transformer-based language model developed by Google that is fine-tuned from the T5 model (Text-To-Text Transfer Transformer). 

It excels at a variety of Natural Language Processing (NLP) tasks, such as question answering, summarization, and translation, because it treats all tasks as text-to-text problems, often achieving strong results even with few-shot learning. 

The "base" version is smaller and more accessible than other Flan-T5 variants, making it suitable for use on personal laptops. 

In [ ]:
# --- PART 2: THE GENERATOR (LLM Setup) ---

# We use Flan-T5-Base because it follows instructions well and runs fast on CPU.
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def retrieve_context(query, k=1):
    """
    1. Embed the query.
    2. Search FAISS for the nearest 'k' document vectors.
    3. Return the text of those documents.
    """
    query_embedding = embedder.encode([query])
    
    # faiss.search takes (query_vectors, k_neighbors)
    # Returns: distances (scores) and indices (indexes of documents)
    distances, indices = index.search(query_embedding, k)
    
    # Fetch the actual text using the indices
    retrieved_docs = [documents[idx] for idx in indices[0]]
    return retrieved_docs

def generate_answer(query):
    """
    1. Retrieve relevant context.
    2. Construct a prompt with that context.
    3. Generate answer.
    """
    # Step A: Retrieve
    context_docs = retrieve_context(query, k=1)
    context_text = " ".join(context_docs)
    
    print(f"[Retrieved Context]: {context_text}")
    
    # Step B: Augment Prompt
    # We explicitly tell the model to use the context.
    prompt = f"Use the following context to answer the question.\nContext: {context_text}\nQuestion: {query}"
    
    # Step C: Generate
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Generate output (max_new_tokens limits the answer length)
    outputs = model.generate(**inputs, max_new_tokens=50)
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# --- PART 3: TESTING ---

print("--- RAG Demo Start ---")

# Test 1: A question that requires the specific knowledge base
query1 = "What is the secret code for the server?"
print(f"[Question]: {query1}")
answer1 = generate_answer(query1)
print(f"[AI Answer]: {answer1}\n")

# Test 2: A question about the "fake" future fact
query2 = "Who is the CEO of TechCorp?"
print(f"[Question]: {query2}")
answer2 = generate_answer(query2)
print(f"[AI Answer]: {answer2}\n")

---

### Using RAG to demonstrate realtime data retrieval, augmentation and generation

This technique demonstrates implementing tools to fetch and source data in realtime

In [ ]:
# Demo on how models hallucinate when prompted without backing knowledge

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = "What is the temperature today in Chennai ?"

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

##### Now let's back this model with relevant information in realtime


In [ ]:
# Create a realtime data retrieval agent using RAG principles
# This agent will fetch live weather data and use it to answer user queries.


# --- PART 1: THE TOOL (Real-Time Retriever) ---

#from dotenv import load_dotenv
import requests
import os
#load_dotenv("../../data/dotenv.sh")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
MOCK_MODE = True  # Set to True if you have a do not have real API Key


def get_weather_tool(city_name):
    """
    Acts as our 'Retriever'. Instead of searching a DB, 
    it hits the API for live data.
    """
    if MOCK_MODE:
        # Simulate an API response for demonstration
        import random
        temps = {"London": 12, "Chennai": 34, "New York": 22, "Antarctica": -30}
        temp = temps.get(city_name, 25) # Default to 25 if unknown
        condition = random.choice(["Rainy", "Sunny", "Cloudy"])
        return f"Current weather in {city_name}: {condition}, Temperature: {temp}°C, Humidity: 60%."

    # Real API Call
    base_url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city_name,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric"
    }
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        if response.status_code == 200:
            weather_desc = data['weather'][0]['description']
            temp = data['main']['temp']
            return f"Current weather in {city_name}: {weather_desc}, Temperature: {temp}°C."
        else:
            return f"Error: Could not find weather for {city_name}."
    except Exception as e:
        return f"Error connecting to weather API: {str(e)}"

get_weather_tool("Chennai")

In [ ]:
# --- PART 2: THE INTELLIGENCE (LLM) ---

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def run_chat_agent(user_query):
    """
    This function simulates a simple 'Agentic' workflow:
    1. DECIDE: Do I need a tool? (Simple keyword check for this demo)
    2. ACT: Call the tool if needed.
    3. OBSERVE: Get the output.
    4. GENERATE: Answer the user.
    """
    
    # 1. Simple "Router" Logic (In production, an LLM decides this step too)
    retrieved_context = ""
    
    if "weather" in user_query.lower() or "temperature" in user_query.lower():
        # Extract city (naive extraction for demo purposes)
        words = user_query.split()
        # Let's assume the last word is the city for simplicity (e.g., "weather in London")
        city = words[-1].strip("?") 
        
        print(f"[Agent]: Detected weather request for '{city}'. Calling Tool...")
        
        # 2. Call the Tool (Retrieval)
        retrieved_context = get_weather_tool(city)
        print(f"[Tool Output]: {retrieved_context}")
    
    else:
        print("[Agent]: General knowledge question.")
        retrieved_context = "No external data retrieved."

    # 3. Augment the Prompt (RAG Step)
    prompt = f"""
    Answer the question based on the context below. If the context has weather data, use it.
    
    Context: {retrieved_context}
    
    Question: {user_query}
    """
    
    # 4. Generate Answer
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    final_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return final_answer


In [ ]:
# --- PART 3: TESTING ---

print("--- Weather Agent Demo ---\n")

# Test 1: Real-time Data Request
q1 = "What is the weather in Chennai?"
print(f"User: {q1}")
ans1 = run_chat_agent(q1)
print(f"AI: {ans1}\n")

print("-" * 30 + "\n")

# Test 2: General Knowledge Request
q2 = "What is the capital of France?"
print(f"User: {q2}")
ans2 = run_chat_agent(q2)
print(f"AI: {ans2}\n")

# Test 3: Another Real-time Data Request
q3 = "Can you tell me the temperature in Delhi?"
print(f"User: {q3}")
ans3 = run_chat_agent(q3)
print(f"AI: {ans3}\n")

#### Implementing proper **tooling** for Agentic AI

In [ ]:
import json
import inspect

# --- PART 1: THE TOOL (The Actual Python Function) ---

def get_weather(city, unit="celsius"):
    """
    Fetches weather data for a specific city.
    """
    # Mock database of weather info
    data = {
        "london": {"temp": 15, "cond": "Cloudy"},
        "paris": {"temp": 18, "cond": "Sunny"},
        "new york": {"temp": 22, "cond": "Rainy"}
    }
    
    city_key = city.lower()
    info = data.get(city_key, {"temp": 20, "cond": "Unknown"})
    
    return json.dumps({
        "city": city,
        "temperature": info["temp"],
        "condition": info["cond"],
        "unit": unit
    })


In [ ]:

# --- PART 2: THE SCHEMA (Teaching the LLM about the tool) ---
# This is what we send to the LLM so it knows HOW to call the function.
weather_tool_schema = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city, e.g. San Francisco"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The temperature unit to use."
                }
            },
            "required": ["city"]
        }
    }
}


In [ ]:

# --- PART 3: THE "BRAIN" (Simulating the LLM) ---

def mock_llm_inference(messages, tools):
    """
    SIMULATION: In a real app, this is where you call:
    openai.chat.completions.create(model="gpt-4", messages=messages, tools=tools)
    
    We simulate the LLM's decision-making logic here.
    """
    last_user_message = messages[-1]['content'].lower()
    
    # SCENARIO A: User asks about weather -> LLM decides to call tool
    if "weather" in last_user_message or "temperature" in last_user_message:
        # The LLM "generates" this JSON structure specifically to call our tool
        return {
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": "call_12345",
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "arguments": '{"city": "London", "unit": "celsius"}' 
                }
            }]
        }
    
    # SCENARIO B: User asks normal question -> LLM answers directly
    else:
        return {
            "role": "assistant",
            "content": "I am a weather bot, but I can also chat! How can I help?"
        }


In [ ]:
# --- PART 4: THE RUNTIME (The Loop) ---

def run_conversation(user_query):
    # 1. Initialize Conversation
    messages = [{"role": "user", "content": user_query}]
    tools = [weather_tool_schema]

    print(f"User: {user_query}")

    # 2. First LLM Call (The Decision)
    print("LLM is thinking...")
    response_message = mock_llm_inference(messages, tools)

    # 3. Check if LLM wants to call a tool
    tool_calls = response_message.get("tool_calls")

    if tool_calls:
        print("LLM decided to call a tool!")
        
        # 4. Execute the Tool(s)
        available_functions = {"get_weather": get_weather}
        
        for tool_call in tool_calls:
            function_name = tool_call['function']['name']
            function_args = json.loads(tool_call['function']['arguments'])
            
            print(f"Calling Function: {function_name} with args {function_args}")
            
            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)
            
            print(f"Tool Output: {function_response}")
            
            # 5. Append Tool Output to Conversation
            # We act as the "tool" role providing the data back to the LLM
            messages.append(response_message) # Add the assistant's "call"
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call['id'],
                "name": function_name,
                "content": function_response
            })

        # 6. Second LLM Call (The Final Answer)
        # In a real app, we send the updated 'messages' list back to the LLM.
        # Since this is a mock, we just print what the LLM would likely say.
        print("\nAI Final Response: 'The current weather in London is Cloudy with a temperature of 15°C.'")
        
    else:
        print(f"AI: {response_message['content']}")


In [ ]:

# --- TEST RUN ---
run_conversation("Can you tell me the weather in London?")

#### Another demo for Tool calling (Agentic AI) for Apple Silicon (MLX)

In [ ]:
%pip install mlx-lm

In [ ]:
import json
from mlx_lm import load, generate
from transformers import AutoTokenizer

# --- 1. SETUP: Load Native Mac Model (4-bit Quantized) ---

# We use a model pre-converted to MLX format (4-bit)
# This downloads ~5GB instead of 15GB and maps directly to Unified Memory.
model_id = "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit"

print(f"Loading {model_id} on Apple Silicon... (First run will download)")

# mlx_lm.load returns the model and tokenizer ready for the M-series chip
model, tokenizer = load(model_id, tokenizer_config={"trust_remote_code": True})

print("Model loaded on Neural Engine/GPU!")


In [ ]:

# --- 2. DEFINE THE TOOLS (Same as before) ---

def get_current_weather(location, unit="celsius"):
    """Get the current weather for a location."""
    # Mock data
    print(f"[Tool Executing] Fetching weather for {location} in {unit}...")
    return json.dumps({
        "location": location,
        "temperature": "22",
        "unit": unit,
        "condition": "Partly Cloudy"
    })

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use. Infer this from the users location.",
                    },
                },
                "required": ["location", "unit"],
            },
        },
    }
]


In [ ]:
# --- 3. THE AGENT LOGIC (Updated for MLX) ---

def run_local_agent_mlx(user_query):
    messages = [{"role": "user", "content": user_query}]

    # A: FORMAT PROMPT
    # We set tokenize=False because MLX's generate() prefers a raw string prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        add_generation_prompt=True,
        tokenize=False 
    )

    # B: GENERATE DECISION
    # mlx_lm.generate is simpler than the PyTorch loop
    response_text = generate(
        model, 
        tokenizer, 
        prompt=prompt, 
        verbose=False, 
        max_tokens=128
    )
    
    print(f"\n[Model Thought]: {response_text}")

    # C: PARSE & EXECUTE (Same Logic)
    tool_call_data = None
    if "get_current_weather" in response_text:
        try:
            start = response_text.find("{")
            end = response_text.rfind("}") + 1
            json_str = response_text[start:end]
            tool_call_data = json.loads(json_str)
        except:
            print("Could not parse JSON from model output.")

    if tool_call_data:
        func_name = tool_call_data.get("name")
        args = tool_call_data.get("parameters") or tool_call_data.get("arguments")
        
        if func_name == "get_current_weather":
            tool_result = get_current_weather(**args)
            
            # D: FEED BACK TO MODEL
            # We need to reconstruct the prompt with the tool output
            # Since MLX generate is stateless, we append to our messages list and re-template
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "tool", "name": func_name, "content": tool_result})
            
            final_prompt = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=False
            )
            
            final_answer = generate(
                model, 
                tokenizer, 
                prompt=final_prompt, 
                verbose=False, 
                max_tokens=128
            )
            
            return final_answer
            
    return response_text


In [ ]:
# --- 4. TEST IT ---

print("--- Local Agent (Apple M4 Edition) ---")
query = "What's the weather like in London today?"
print(f"User: {query}")

answer = run_local_agent_mlx(query)
print(f"AI: {answer}")

---

In [ ]:
def classify_sentiment(text):
    from transformers import pipeline
    classifier = pipeline("sentiment-analysis", model="bert-base-uncased")
    result = classifier(text)[0]
    label, score = result['label'], result['score']

    output = {"LABEL_0": "❌", "LABEL_1": "✅"}

    print(f"{output[label]}:{label}: {score:.4f}")
    return {output[label]: score}

classify_sentiment("The movie was amazing!")